# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a reproducible guide for loading, exploring, and processing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

The dataset is defined via a Croissant schema with multiple record sets, fields, and columns, each referenced by their unique `@id`.

### Dataset Source
The source Croissant schema is available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure the mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the provided Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant URL
CROISSANT_URL = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(CROISSANT_URL)
# To display all metadata fields, convert to json and pretty-print (optional):
meta_dict = dataset.metadata.to_json()
print(f"{meta_dict['name']}: {meta_dict['description']}")

# Let's look at dataset identifier, version, citation, and keywords as well
print(f"Identifier: {meta_dict['identifier']}")
print(f"Version:    {meta_dict['version']}")
print(f"Dataset keywords: {', '.join(meta_dict['keywords'])}")
print(f"Date Published: {meta_dict['datePublished']}")
print(f"Cite as: {meta_dict['citeAs']}")

## 2. Data Overview

List available record sets, each of their fields, field IDs (`@id`), and the underlying columns. All IDs are referenced by their `@id` per Croissant specification.

This lets us see the data's structure before loading any records.


In [ ]:
print("Available record sets:")
record_sets = dataset.record_sets
for recs in record_sets:
    print(f"\nRecord set: {recs['@id']}")
    print(f"  Name / label: {recs.get('name', recs.get('label', 'N/A'))}")
    print(f"  Description: {recs.get('description', 'N/A')}")

    # Print all fields for this record set
    fields = recs['field'] if isinstance(recs['field'], list) else [recs['field']]
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            field_id = field['@id']
            field_label = field.get('name', field.get('label', ''))
            print(f"    - {field_id}: {field_label}")
        else:
            print(f"    - {field}")
    # Print columns info
    if 'column' in recs:
        columns = recs['column'] if isinstance(recs['column'], list) else [recs['column']]
        print("  Columns:")
        for column in columns:
            if isinstance(column, dict):
                column_id = column['@id']
                column_label = column.get('name', column.get('label', ''))
                print(f"    - {column_id}: {column_label}")
            else:
                print(f"    - {column}")

## 3. Data Extraction

We select one or more specific record sets by their `@id`, and load records as DataFrames for analysis. All processing will reference the appropriate `@id` throughout.


In [ ]:
# For this dataset, there is usually one main record set for the protein table.
# Let's discover it dynamically from the schema by looking for the main table.

# The main record set most often contains 'protein' or 'abundance' in its @id or label.
main_rs_id = None
for recs in dataset.record_sets:
    idval = recs['@id']
    nameval = recs.get('name', recs.get('label', '')).lower()
    descval = recs.get('description', '').lower()
    if 'protein' in idval.lower() or 'protein' in nameval or 'abundance' in nameval or 'abundance' in descval:
        main_rs_id = idval
        break

if main_rs_id is None:
    # Fallback to first record set
    main_rs_id = dataset.record_sets[0]['@id']

print(f"Main record set selected: {main_rs_id}\n")

# In this notebook, we'll focus on the main record set, but you can list all IDs for expansion:
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show column names for the selected main record set
print(f"Columns in {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's process the protein abundance data. We'll filter for high abundance, normalize a key numeric variable, and group by one attribute if available.

All variables are referenced by their `@id`. You can inspect DataFrame's columns to choose more fields.


In [ ]:
# Find a numeric field in columns (e.g., abundance, peptide count, MW). Adjust @id as needed.
import numpy as np

# Attempt to select commonly available numeric fields
candidate_numeric_ids = [
    'cr:abundance', 'cr:normalized_abundance', 'cr:peptide_count', 'cr:mw', 'cr:molecular_weight', 'cr:coverage_percentage', 'cr:pi'
]
numeric_field = None
# Choose first matching column as numeric_field
for f in dataframes[main_rs_id].columns:
    if f in candidate_numeric_ids and np.issubdtype(dataframes[main_rs_id][f].dtype, np.number):
        numeric_field = f
        break
    # Fallback: try columns ending with _abundance, _count, _mw, etc.
    if any(k in f.lower() for k in ['abundance', 'count', 'mw', 'weight', 'coverage', 'pi']):
        if np.issubdtype(dataframes[main_rs_id][f].dtype, np.number):
            numeric_field = f
            break
if numeric_field is None:
    raise Exception("Couldn't automatically find a numeric field. Please check column names.")
print(f"Selected numeric field: {numeric_field}")

# Filtering: Pick reasonable threshold depending on field distribution
# We'll filter for values above the median
threshold = float(dataframes[main_rs_id][numeric_field].median())
filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (median): {len(filtered_df)} rows")
print(filtered_df.head())

# Normalize the selected numeric field (mean 0, std 1)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical variable, e.g. by sample or protein type
group_field = None
# Try common choices
for c in dataframes[main_rs_id].columns:
    if any(x in c.lower() for x in ['type', 'class', 'gene', 'sample', 'protein', 'group']):
        if not np.issubdtype(dataframes[main_rs_id][c].dtype, np.number):
            group_field = c
            break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field}, mean {numeric_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical group field found.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field (e.g., protein abundance), and if grouping is available, plot group means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Distribution plot
plt.figure(figsize=(8,4))
sns.histplot(dataframes[main_rs_id][numeric_field], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Boxplot for normalized field
plt.figure(figsize=(4,6))
sns.boxplot(y=filtered_df[f"{numeric_field}_normalized"])
plt.title(f"Normalized {numeric_field} (filtered)")
plt.show()

# If group_field and grouped_df, show barplot
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10,4))
    # Only plot if enough unique values
    plot_df = grouped_df.copy().sort_values(numeric_field, ascending=False)
    if len(plot_df) < 50:
        sns.barplot(data=plot_df, x=group_field, y=numeric_field, palette="viridis")
        plt.xticks(rotation=60)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load a FAIR-certified Croissant dataset using `mlcroissant`, inspect its structure, access fields and record sets by their `@id`, and perform common preprocessing and exploratory analysis tasks. 

- The dataset contains mass spectrometry-based quantification of proteins from human extracellular vesicles.
- We extracted data using the record set's unique `@id`, performing filtering, normalization, and visualization of numeric variables such as abundance or peptide count.
- The Croissant metadata facilitates data discovery and programmatic access in a reproducible, machine-readable way. 

For other biomedical or omics datasets, you can adapt this notebook by changing only the Croissant schema URL and parameterizing the field selection.
